In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from gen_function_make_geometry_idf_sample_initial import make_geometry_idf_sample_distr
from ep_function_make_enriched_idf import make_enriched_idf
from ep_function_run_simulations import function_run_simulations
from ep_function_read_output import function_read_output

from surr_function_make_input_vector_and_predict import function_make_input_vector_and_predict
from surr_function_make_input_vector_and_predict_over_time import function_make_input_vector_and_predict_over_time

from scipy.stats import t
from vis_function_building_assessment_longterm import function_building_assessment_longterm
from vis_function_neighborhood_maps import function_neighborhood_maps
from vis_function_cumulative_sums import calculate_mean_ci_by_year, function_cumulative_sums_neighborhood

In [ ]:
# Sample for a specific BAG Pand ID (retrieve based on the building address from: https://bagviewer.kadaster.nl/lvbag/bag-viewer)
bag_pand_id ='["0599100000614202"]'
n_sim=2
(idf, building, records_init_singleb) = make_geometry_idf_sample_distr(bag_pand_id, n_sim)

In [ ]:
# EnergyPlus path
# Enrich the idf files based on the properties of each path, run the simulations & read the output
output_dir = "./simulation_files"
epw_path = './INP_epw_file/NLD_ZH_Rotterdam.The.Hague.AP.063440_TMYx.2009-2023.epw'

make_enriched_idf(idf, building, records_init_singleb)
function_run_simulations(output_dir, epw_path)
df_records_ep = function_read_output("./simulation_files", records_init_singleb) 
# Heating outputs & Underheating hours are calculated for heating season (October to March)
# Cooling outputs & Overheating hours are calculated for cooling season (April to September)

In [ ]:
# Surrogate path - Assess building performance on year 0
# Create input vector, assess for building performance
records_res_singleb = function_make_input_vector_and_predict(idf, building, records_init_singleb)

In [ ]:
# Surrogate path - Assess performance for building lifetime
# Connect with time-dependent degradation models & retrofit plans, Create input vector & Assess performance along building lifetime

building_lifetime = 70

# Create retrofit plan (Initially do-nothing for all years and then change strategy for specific component and year)
components = ["wall", "roof", "floor", "window", "heating_system", "cooling_system", "inf_rate"]
retrofit_plan = {component: ["do_nothing"] * 70 for component in components}
retrofit_plan["wall"][1] = "retrofit_6.2"
retrofit_plan["roof"][1] = "retrofit_8.3"
retrofit_plan["floor"][1] = "retrofit_5.7"
retrofit_plan["window"][1] = "retrofit_3"
retrofit_plan["inf_rate"][1] = "retrofit_0.0004"
retrofit_plan["heating_system"][1] = "retrofit_heat_pump"
retrofit_plan["cooling_system"][1] = "retrofit_dx_airco"

records_res_singleb_time = function_make_input_vector_and_predict_over_time(idf, building, records_init_singleb, building_lifetime, retrofit_plan)

In [ ]:
# Visualize probabilistic assessment for single building in the long term
function_building_assessment_longterm(records_res_singleb_time, bag_pand_id)

In [ ]:
# Surrogate path - Assess performance for multiple buildings in the long-term

# Output columns
output_cols = [
    "heating_dem_m2",
    "cooling_dem_m2",
    "heating_gas_m2",
    "heating_elec_m2",
    "heating_distr_heat_m2",
    "cooling_elec_m2",
    "underheat_m2",
    "overheat_m2",
    "carbon_emissions"
]

# Create a retrofit plan (Initially do-nothing for all years and then change strategy for specific component and year)
components = ["wall", "roof", "floor", "window", "heating_system", "cooling_system", "inf_rate"]
retrofit_plan = {component: ["do_nothing"] * 70 for component in components}
retrofit_plan["wall"][1] = "retrofit_6.2"
retrofit_plan["roof"][1] = "retrofit_8.3"
retrofit_plan["floor"][1] = "retrofit_5.7"
retrofit_plan["window"][1] = "retrofit_3"
retrofit_plan["inf_rate"][1] = "retrofit_0.0004"
retrofit_plan["heating_system"][1] = "retrofit_heat_pump"
retrofit_plan["cooling_system"][1] = "retrofit_dx_airco"

# Read file with building IDs to test (here they belong to the same neighborhood)
gdf_pands = pd.read_csv("pands_kralingseveer.csv")
n_buildings = len(gdf_pands)

n_samples = 5
n_years =  70
year_values = list(range(0, n_years)) 
n_outputs = len(output_cols)

# Initiate the numpy arrays to collect the outputs per building, per year
mean_by_building_year_output = np.zeros((n_buildings, n_years, n_outputs), dtype=np.float32)
lower_ci_by_building_year_output = np.zeros((n_buildings, n_years, n_outputs), dtype=np.float32)
upper_ci_by_building_year_output = np.zeros((n_buildings, n_years, n_outputs), dtype=np.float32)
se_by_building_year_output = np.zeros((n_buildings, n_years, n_outputs), dtype=np.float32)

# Predict performance per building and calculate mean and confidence intervals per year
for b_idx, row in gdf_pands.iterrows():
    raw_id = str(row["bag_pand_id"])
    (idf, building, records_init) = make_geometry_idf_sample_distr(raw_id, n_samples)
    records_res_time = function_make_input_vector_and_predict_over_time(idf, building, records_init, 70, retrofit_plan)
    df_records_res_time = pd.DataFrame(records_res_time)
    stats = calculate_mean_ci_by_year(df_records_res_time, output_cols, year_values, confidence=0.95)
    
    mean_by_building_year_output[b_idx, :, :] = stats["mean"]
    lower_ci_by_building_year_output[b_idx, :, :] = stats["lower"]
    upper_ci_by_building_year_output[b_idx, :, :] = stats["upper"]
    se_by_building_year_output[b_idx, :, :] = stats["se"]

# Compute sum of means at neighborhood scale and confidence intervals 
t_crit = t.ppf(0.975, n_samples)
city_sum_mean = np.sum(mean_by_building_year_output, axis=0)
city_se = np.sqrt(np.sum(se_by_building_year_output ** 2, axis=0))
city_ci_half_width = t_crit * city_se
city_lower = city_sum_mean - city_ci_half_width
city_upper = city_sum_mean + city_ci_half_width

In [ ]:
# Create neighborhood maps for each output variable showing performance with building-level resolution
year_idx = 0
function_neighborhood_maps(gdf_pands, mean_by_building_year_output, year_idx)

In [ ]:
# Visualize cumulative sum and confidence intervals for the whole neighborhood
function_cumulative_sums_neighborhood(city_sum_mean, city_lower, city_upper, n_years)